# CASMI 2026 — FPNet

A **launcher, not a pipeline**, and an **experiment, not an improvement**.

The analogue-propagation submission scored **0.080** on the board — the random floor —
while scoring 1.0000 on the visible `test.parquet`, whose spectra turn out to be exact
copies of training spectra. So Kaggle re-runs this notebook against hidden data, and on
that data a method that depends on finding a similar spectrum in the library carries no
information.

FPNet depends on no library at all. It predicts a molecular fingerprint from the spectrum
and compares it against each candidate's exact RDKit fingerprint, so it can score a
structure nothing has ever measured. That makes this submission the direct test of which
hypothesis survives:

| board score | what it means |
|---|---|
| ≫ 0.080 | the failure was library dependence, and the learned signal transfers |
| ≈ 0.080 | the answer is probably not in our candidate bank at all, and rebuilding it from external structure databases becomes the priority |

Offline, on 3,200 molecules the epoch selection never saw: MRR 0.7998, top-1 68.3 %,
answer in the 25 for 99.0 %, per-bit AUC 0.917. **That number comes from the same
validation set the board just invalidated**, so it is quoted as context, not as a
prediction. Its own two biases are known: the answer was always in the candidate bank,
and FPNet trained on the validation molecules' relatives.

`DIAG` lines print into this run's log — unparseable adducts, window sizes, the model's
provenance — so the result explains itself whatever it is.


In [ ]:
import os, sys, time, zipfile
from pathlib import Path

T0 = time.time()
# Overridable so the mounts can be faked locally and these exact cells rehearsed before
# a run is queued -- scripts/rehearse_kernel.py. On Kaggle the variable is unset.
INPUT = Path(os.environ.get("KAGGLE_INPUT", "/kaggle/input"))


def log(message):
    print(f"[{time.time() - T0:7.1f}s] {message}", flush=True)


def find(marker):
    # First directory holding a file named `marker`. Searched by content rather than by
    # path: Kaggle names a mount after its dataset, and a rename would otherwise break
    # this notebook in silence.
    for current, _, files in os.walk(INPUT):
        if marker in files:
            return Path(current)
    return None


def find_library():
    # The directory to put on sys.path so `import casmi` resolves.
    #
    # THREE mount shapes have to be handled, and which one arrives is Kaggle's decision,
    # not ours. All three were observed:
    #   1. the package expanded with its directory, `.../casmi/submit.py`
    #   2. `casmi.zip`, which the upload produces and Kaggle sometimes serves as-is
    #   3. **flattened** -- the ten modules at the dataset root with no `casmi/` at all,
    #      which is what the first real push actually produced. `import casmi` cannot
    #      work in that shape, so the package directory is rebuilt here.
    #
    # Everything is materialised under ./lib, relative to the working directory: that is
    # /kaggle/working on Kaggle and the repo locally, so every branch is rehearsable.
    # A hardcoded /kaggle/working was uncreatable elsewhere and hid a broken branch.
    root = find("submit.py")
    if root is not None and root.name == "casmi":
        return root.parent

    for current, _, files in os.walk(INPUT):
        if "casmi.zip" in files:
            dest = Path("lib").resolve()
            dest.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(Path(current) / "casmi.zip") as z:
                z.extractall(dest)
            return dest

    if root is not None:
        import shutil
        dest = Path("lib/casmi").resolve()
        dest.mkdir(parents=True, exist_ok=True)
        modules = sorted(root.glob("*.py"))
        for module in modules:
            shutil.copy2(module, dest / module.name)
        log(f"rebuilt the package from {len(modules)} flattened modules")
        return dest.parent
    return None


library = find_library()
if library is None:
    raise SystemExit("the casmi-src dataset is not mounted")
sys.path.insert(0, str(library))
log(f"library: {library}")

comp = find("train.parquet")
if comp is None:
    raise SystemExit("the competition data is not mounted")
log(f"competition data: {comp}")
print(sorted(p.name for p in comp.iterdir()))

In [ ]:
# RDKit is the one hard dependency: it computes every candidate fingerprint and every
# InChIKey the metric compares. The Kaggle image does not ship it -- the first run died
# here at 0.8s with ModuleNotFoundError, which is exactly why this check is at the top
# and unguarded rather than after an hour of parquet reading.
#
# Installed from the wheel the competition itself publishes (`metric/rdkit-2026-3-3-wheel`
# -- the `metric` account is Kaggle's own, the one that also publishes the official
# scorer). Pinning 2026.03.3 is correctness, not superstition: the scorer computes
# InChIKeys with it and ours have to match.
import glob
import subprocess

PINNED = "2026.03.3"


def installed_rdkit():
    # A capability probe, not a swallowed error: absent is an answer, and the unguarded
    # import below is what fails the run if the install did not work.
    try:
        import rdkit
        return rdkit.__version__
    except ImportError:
        return None


have = installed_rdkit()
if have != PINNED:
    # Matched on the interpreter tag alone, NOT on the version string. The wheel is named
    # `rdkit-2026.3.3-cp313-...` while `rdkit.__version__` reports `2026.03.3`, and a glob
    # built from the padded form matched nothing -- which cost a run, because the error
    # message then asserted "no wheel is mounted" when the wheel was mounted and the
    # pattern was wrong.
    tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    wheels = sorted(glob.glob(str(INPUT / "**" / f"rdkit-*-{tag}-*.whl"), recursive=True))
    if wheels:
        log(f"rdkit {have or 'absent'} -> installing {Path(wheels[0]).name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index",
                               "--quiet", wheels[0]])
    elif have is None:
        # Show the evidence rather than a conclusion: every wheel actually visible, so a
        # repeat failure is diagnosable from one log instead of three runs.
        seen = sorted(glob.glob(str(INPUT / "**" / "*.whl"), recursive=True))
        raise SystemExit(
            f"rdkit is absent and no {tag} wheel matched. Wheels under {INPUT}:\n  "
            + ("\n  ".join(seen) if seen else "(none)"))
    else:
        # Local rehearsal runs whatever the venv has; the difference is logged, not hidden.
        log(f"WARNING: rdkit {have} != pinned {PINNED}, and no wheel matched")

import numpy, pandas, scipy, rdkit
from rdkit import Chem

log(f"python {sys.version.split()[0]}  numpy {numpy.__version__}  "
    f"pandas {pandas.__version__}  scipy {scipy.__version__}  rdkit {rdkit.__version__}")
log(f"cpus: {os.cpu_count()}")
assert Chem.MolFromSmiles("CCO") is not None, "RDKit cannot parse ethanol"
assert Chem.MolToInchiKey(Chem.MolFromSmiles("CCO")).split("-")[0] == "LFQSCWFLJHTTHZ", \
    "RDKit computes an unexpected InChIKey for ethanol"

In [ ]:
from casmi import submit as S

checkpoint = find("fpnet.pt")
if checkpoint is None:
    raise SystemExit("the casmi-fpnet dataset is not mounted")
log(f"checkpoint: {checkpoint}")

submission = S.run_submission(
    train_path=comp / "train.parquet",
    test_path=comp / "test.parquet",
    out_path="submission.csv",
    method="fpnet",
    fpnet_checkpoint=checkpoint / "fpnet.pt",
)


In [ ]:
import pandas as pd

# Assert the shape Kaggle expects. A malformed file scores zero without saying why, and
# the failure is indistinguishable from a method that does not work.
test_ids = pd.read_parquet(comp / "test.parquet", columns=["molecule_id"]).molecule_id
S.check_submission(submission, test_ids)
log("done")
submission.head()